In [5]:
# baseline.ipynb - classical ML baseline (Random Forest / SVM) for AI music detection

import os
import numpy as np
import pandas as pd
import librosa

# load the split table (same CSV the preprocessing used)
splits = pd.read_csv("data/subset_splits.csv")

print("Loaded splits:", splits.shape)

Loaded splits: (4000, 22)


In [6]:
def extract_features(file_path, clip_seconds=15):

    """Load an audio file, take a centred 15s clip and return a 1D feature vector
       of summary audio features (MFCCs + spectral features) Returns None if too short"""

    # load the audio (y = waveform, sr = sample rate)
    y, sr = librosa.load(file_path)

    clip_len = clip_seconds * sr # convert clip length to total number of samples

    if len(y) < clip_len: # skip if its too short

        return None
    
    # take the centred 15s clip (same logic as get_clip in the preprocessing)

    middle = len(y) // 2 # centre sample of the track

    half = clip_len // 2 # half a clip to put on both sides of the middle

    clip = y[middle - half : middle + half] # this slices out the centred clip

    # extract features from the clip

    # MFCCs this is 13 numbers that summarise the overall "shape" of the sound (timbre) one value per time frame. a (13, 646) grid
    mfccs = librosa.feature.mfcc(y=clip, sr=sr, n_mfcc=13)

    # each MFCC changes over time, so summarise each one by its mean and standard deviation
    mfcc_mean = np.mean(mfccs, axis=1)

    mfcc_std  = np.std(mfccs, axis=1)

    # another 2 extra classic features (also summarised to mean)
    spectral_centroid = np.mean(librosa.feature.spectral_centroid(y=clip, sr=sr)) # brightness of the sound

    zero_crossing     = np.mean(librosa.feature.zero_crossing_rate(y=clip)) # noisiness of the sound

    # stick them all together into one flat feature vector (values for the 13 means, 13 standard deviation and the 2 extras)
    features = np.concatenate([mfcc_mean, mfcc_std, [spectral_centroid, zero_crossing]])

    return features

In [9]:
# test the feature extractor on one real file
test_path = "data/audio/real/real_22920.wav"

feats = extract_features(test_path)

print("Feature vector length:", len(feats))

print("First few values:", feats[:5])

Feature vector length: 28
First few values: [ 24.0431118   83.33361816 -21.65079498  15.48574734   2.14156842]


In [ ]:
def build_audio_path(filename):

    """Build the full path to an audio file from its CSV filename.
       Fakes are .mp3 in data/audio/fake and reals are .wav in data/audio/real."""
    

    # fake filenames start with "fake_" so use the fake folder + .mp3
    if filename.startswith("fake_"):

        return os.path.join("data/audio/fake", filename + ".mp3")
    
    # otherwise it's a real track so use the real folder + .wav
    else:
        
        return os.path.join("data/audio/real", filename + ".wav")

In [ ]:
import os

# lists to collect features (X) and labels (y) as we go. x = feature rows and y = labels ( 0 for real and 1 for fake)
X = []
y = []

# also track which rows that were skipped like in preprocessing
skipped_missing = 0

skipped_short = 0

# loop over every track in the split table
for i, row in splits.iterrows():

    filename = row["filename"]
    
    label = row["target"]  # 0 = real 1 = fake

    audio_path = build_audio_path(filename)   # reuse the path helper from preprocessing

    # skip if the audio file isn't on disk (failed download)
    if not os.path.exists(audio_path):

        skipped_missing += 1

        continue

    # extract the 28 features for this track
    features = extract_features(audio_path)

    # skip if it came back None (too short)
    if features is None:

        skipped_short += 1

        continue

    # keep this tracks features and its label (they stay lined up)
    X.append(features)

    y.append(label)

# final summary of the run
print(f"Collected {len(X)} feature rows, skipped {skipped_missing} missing, {skipped_short} too short")

Collected 3722 feature rows, skipped 277 missing, 1 too short
